## Antyanuprasa

In [19]:
import re

def _normalize_phonetic_sound(akshara):
    """Normalizes Telugu aksharas by mapping sound-equivalent consonants """
    mapping = {
        'శ': 'స',
        'ళ': 'ల',
        'ఱ': 'ర',
        'ణ': 'న',
        'మ్': 'ం',  # nasalization
    }
    normalized = ""
    for char in akshara:
        normalized += mapping.get(char, char)
    return normalized

def _get_telugu_aksharas(text):
    """Splits Telugu text into orthographic syllables (Aksharas) and normalizes sound equivalents."""
    aksharas = []
    current_akshara = ""
    
    for char in text:
        code = ord(char)
        is_modifier = (
            code == 0x0C02 or 
            code == 0x0C03 or 
            (0x0C3E <= code <= 0x0C4D) or 
            (0x0C55 <= code <= 0x0C56)
        )
        
        if is_modifier and current_akshara:
            current_akshara += char
        else:
            if current_akshara:
                aksharas.append(_normalize_phonetic_sound(current_akshara))
            current_akshara = char
            
    if current_akshara:
        aksharas.append(_normalize_phonetic_sound(current_akshara))
    
    return [a for a in aksharas if re.search(r'[\u0C00-\u0C7F]', a)]

def _check_antyanuprasa(lines):
    if len(lines) < 2:
        return None
        
    results = []
    for i in range(len(lines) - 1):
        aksharas1 = _get_telugu_aksharas(lines[i])
        aksharas2 = _get_telugu_aksharas(lines[i+1])
        
        if not aksharas1 or not aksharas2:
            continue
            
        # Find maximum consecutive matching syllables from the end (right to left)
        match_count = 0
        min_len = min(len(aksharas1), len(aksharas2))
        
        for j in range(1, min_len + 1):
            if aksharas1[-j] == aksharas2[-j]:
                match_count += 1
            else:
                break
                
        if match_count > 0:
            matched_1 = aksharas1[-match_count:]
            matched_2 = aksharas2[-match_count:]
            results.append({
                "alankaram": "Antyanuprasa Alankaram",
                "trigger": f"Max matching ending syllables ({match_count}): '{' '.join(matched_1)}' & '{' '.join(matched_2)}'",
                "position": f"Lines {i+1} and {i+2}",
                "match_length": match_count
            })
            
    return results if results else None

In [20]:
antyanuprasa_examples = [
        '''శ్రీరఘురామ !
        చారుతులసీ దళధామ !''',
        '''కరుణా పయోనిధీ !
        శరణాగత విరోధీ !''',
        '''ధరలోని శౌర్యము !
        కరమొప్పు ధైర్యము !''',
        '''కువలయ విలాస !
        భవరోగ వినాశ !''',
        '''సరసిజ నయనా !
        సురనుత లలనా !''',
        '''మురమథన కంసా !
        దరహాస ప్రకాశా !''',
        '''అమరవర వందితా !
        సమరవిజయ గీతా !''',
        '''ధరణీరమణ రాఘవా !
        కరుణాలవాల మేఘా !''',
        '''విమలగుణ ధామా !
        కమలాక్ష రామారామా !''',
        '''శ్రీకర భూపా !
        లోకవిలాపా !'''
    ]

for example in antyanuprasa_examples:
    lines = example.split('\n')
    result = _check_antyanuprasa(lines)
    if result:
        print(f"Example:\n{example}\nDetected Alankaram:\n{result}\n")
    else:
        print(f"Example:\n{example}\nNo Alankaram detected.\n")

Example:
శ్రీరఘురామ !
        చారుతులసీ దళధామ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': "Max matching ending syllables (1): 'మ' & 'మ'", 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
కరుణా పయోనిధీ !
        శరణాగత విరోధీ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': "Max matching ending syllables (1): 'ధీ' & 'ధీ'", 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
ధరలోని శౌర్యము !
        కరమొప్పు ధైర్యము !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': "Max matching ending syllables (3): 'ర్ య ము' & 'ర్ య ము'", 'position': 'Lines 1 and 2', 'match_length': 3}]

Example:
కువలయ విలాస !
        భవరోగ వినాశ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': "Max matching ending syllables (1): 'స' & 'స'", 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
సరసిజ నయనా !
        సురనుత లలనా !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigg